In [12]:
! source /home/jupyter-r.yazdi/myproject/ryazdi/dti3d/bin/activate

In [1]:
import os
import shutil
#shutil.rmtree('PBC-YOLO-811')

In [16]:
import cv2
import glob
import os
import numpy as np

# Paths
original_train_images_dir = "/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/PBC-YOLO-811/images/val/"
clahe_train_images_dir = "/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/CLAHE/PBC-YOLO-811/images/val/"

print(f"✓ Original images directory: {original_train_images_dir}")
print(f"✓ CLAHE output directory: {clahe_train_images_dir}")

# Create output directory if missing
os.makedirs(clahe_train_images_dir, exist_ok=True)

✓ Original images directory: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/PBC-YOLO-811/images/val/
✓ CLAHE output directory: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/CLAHE/PBC-YOLO-811/images/val/


In [17]:
# CLAHE parameters (tune based on your blood smear characteristics)
CLAHE_CLIP_LIMIT = 2.0      # Higher = more contrast (try 1.5-4.0)
TILE_GRID_SIZE = (8, 8)     # Smaller tiles = more local adaptation (try 4x4 to 16x16)

# Initialize CLAHE object (applied to L channel only)
clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=TILE_GRID_SIZE)

# Get all image paths (multiple extensions)
image_paths = []
for ext in ['*.jpg']:
    image_paths.extend(glob.glob(os.path.join(original_train_images_dir, ext)))

print(f"\nFound {len(image_paths)} images to process...")

if not image_paths:
    print(f"❌ No images found in {original_train_images_dir}")
    print("Please verify the path and image extensions (.jpg)")
else:
    processed_count = 0
    failed_count = 0
    
    for img_path in image_paths:
        try:
            # Read original color image (BGR format)
            bgr_img = cv2.imread(img_path)
            if bgr_img is None:
                raise ValueError("cv2.imread returned None")

            # Convert to LAB color space
            lab_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
            l_channel, a_channel, b_channel = cv2.split(lab_img)

            # Apply CLAHE ONLY to L (lightness) channel
            clahe_l = clahe.apply(l_channel)

            # Merge enhanced L channel with original color channels
            merged_lab = cv2.merge([clahe_l, a_channel, b_channel])

            # Convert back to BGR for saving
            clahe_bgr = cv2.cvtColor(merged_lab, cv2.COLOR_LAB2BGR)

            # Save result
            file_name = os.path.basename(img_path)
            output_path = os.path.join(clahe_train_images_dir, file_name)
            cv2.imwrite(output_path, clahe_bgr)

            processed_count += 1
            if processed_count % 5000 == 0:
                print(f"  Processed {processed_count}/{len(image_paths)} images...")

        except Exception as e:
            print(f"⚠️  Failed to process {os.path.basename(img_path)}: {str(e)}")
            failed_count += 1
            continue

    # Final report
    print("\n" + "="*60)
    print(f"✅ CLAHE processing completed")
    print(f"   • Successfully processed: {processed_count}/{len(image_paths)}")
    print(f"   • Failed: {failed_count}")
    print(f"   • Output directory: {clahe_train_images_dir}")
    print(f"\n🔬 Color-preserving CLAHE applied to L channel in LAB space")
    print(f"   • Clip limit: {CLAHE_CLIP_LIMIT}")
    print(f"   • Tile grid size: {TILE_GRID_SIZE}")
    print("="*60)


Found 1687 images to process...

✅ CLAHE processing completed
   • Successfully processed: 1687/1687
   • Failed: 0
   • Output directory: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/CLAHE/PBC-YOLO-811/images/val/

🔬 Color-preserving CLAHE applied to L channel in LAB space
   • Clip limit: 2.0
   • Tile grid size: (8, 8)
